1. Установка зависимостей

In [1]:
# ВАЖНО: Сначала выполните "Среда выполнения → Перезапустить среду выполнения"
# чтобы сбросить сломанные импорты

# Устанавливаем ТОЛЬКО недостающие библиотеки, НЕ трогая torch/torchvision
!pip install -q --upgrade \
    "transformers>=4.46.0" \
    "peft>=0.14.0" \
    "accelerate>=1.2.0" \
    "bitsandbytes>=0.45.0" \
    "datasets>=3.2.0" \
    "trl>=0.13.0" \
    "tokenizers>=0.21.0" \
    "huggingface-hub>=0.27.0" \
    "fsspec>=2024.0"

# Проверяем, что torchvision уже установлен (он есть в Colab по умолчанию)
import torchvision
print(f"torchvision version: {torchvision.__version__}")

print("ГОТОВО. Перезапустите runtime и переходите к ячейке 2.")

torchvision version: 0.26.0+cu128
ГОТОВО. Перезапустите runtime и переходите к ячейке 2.


In [2]:
from google.colab import drive
import os
import shutil

# 1. Монтируем Google Drive
drive.mount('/content/drive')

GDRIVE_COCO_ZIP = '/content/drive/MyDrive/coco_train2017.zip'
LOCAL_COCO_DIR = '/content/coco'

# 2. Скачиваем в Google Drive только если архива там еще нет
if not os.path.exists(GDRIVE_COCO_ZIP):
    print("Архив не найден в Google Drive. Инициализация скачивания...")
    !wget -q http://images.cocodataset.org/zips/train2017.zip -O {GDRIVE_COCO_ZIP}
    print("Скачивание завершено.")
else:
    print("Архив уже существует в Google Drive. Пропуск скачивания.")

# 3. Копируем на локальный SSD Colab и распаковываем
# Это критически важно для скорости I/O во время обучения
if not os.path.exists(LOCAL_COCO_DIR):
    print("Копирование архива на локальный SSD и распаковка (займет ~2-3 минуты)...")
    shutil.copy(GDRIVE_COCO_ZIP, '/content/train2017.zip')
    !unzip -q /content/train2017.zip -d {LOCAL_COCO_DIR}
    !rm /content/train2017.zip
    print("Готово. Изображения на локальном SSD.")
else:
    print("Изображения уже распакованы на локальном SSD.")

Mounted at /content/drive
Архив не найден в Google Drive. Инициализация скачивания...
Скачивание завершено.
Копирование архива на локальный SSD и распаковка (займет ~2-3 минуты)...
Готово. Изображения на локальном SSD.


2. Импорт библиотек и проверка GPU

In [11]:
import torch
print(f"torch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

from transformers import (
    Qwen2VLForConditionalGeneration,
    AutoProcessor,
    TrainingArguments,
    BitsAndBytesConfig,
    Trainer
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import load_dataset

print("Все импорты успешны")
from PIL import Image
import os
import requests
from io import BytesIO
import json
import re

print(f"GPU доступен: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")


torch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
Все импорты успешны
GPU доступен: True
GPU: Tesla T4


In [4]:
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

VRAM: 15.64 GB


3. Загрузка и подготовка данных

In [5]:
# Ячейка 5: Загрузка и подготовка данных (1000 примеров)
print("Загрузка датасетов...")

train_dataset_full = load_dataset("deepvk/LLaVA-Instruct-ru", split="train")
print(f"Всего записей в llava-instruction-ru: {len(train_dataset_full)}")

NUM_TRAIN_SAMPLES = 1000  # Увеличено до 1000
train_dataset = train_dataset_full.shuffle(seed=42).select(range(NUM_TRAIN_SAMPLES))
print(f"Используем для обучения: {len(train_dataset)} записей")

eval_dataset = load_dataset("deepvk/mmbench-ru", split="dev")
print(f"Записей для оценки (mmbench-ru): {len(eval_dataset)}")

Загрузка датасетов...


README.md:   0%|          | 0.00/2.67k [00:00<?, ?B/s]

llava_instruct_ru_train.json: reconstructing file:   0%|          |  0.00B /  197MB            

llava_instruct_ru_train.json: downloading bytes:           |  0.00B            

llava_instruct_ru_val.json: reconstructing file:   0%|          |  0.00B / 77.7MB            

...

README.md:   0%|          | 0.00/3.30k [00:00<?, ?B/s]

mmbench_ru_dev.parquet: reconstructing file:   0%|          |  0.00B / 92.7MB            

mmbench_ru_dev.parquet: downloading bytes:           |  0.00B            

Generating dev split:   0%|          | 0/3910 [00:00<?, ? examples/s]

Записей для оценки (mmbench-ru): 3910


4. Предобработка данных для обучения

In [6]:
# Ячейка 6: Предобработка данных для обучения
print("Проверка типа изображения в датасете:")
print(f"Тип example['image']: {type(train_dataset[0]['image'])}")

if isinstance(train_dataset[0]['image'], str):
    print("Декодирование изображений из строк в PIL Image...")
    from datasets import Image
    train_dataset = train_dataset.cast_column("image", Image())
    print(f"После декодирования тип: {type(train_dataset[0]['image'])}")

def format_conversations(example):
    messages = []
    for turn in example['conversations']:
        role = 'user' if turn['from'] == 'human' else 'assistant'
        text = turn['value'].replace('<image>', '').strip()
        messages.append({"role": role, "content": text})
    return {"messages": messages}

print("Форматирование датасета (без вложенных изображений)...")
train_dataset_formatted = train_dataset.map(
    format_conversations,
    remove_columns=['conversations', 'type', 'id'],
    num_proc=2
)

print("Форматирование завершено успешно.")
print(f"Колонки датасета: {train_dataset_formatted.column_names}")
print(f"Тип image: {type(train_dataset_formatted[0]['image'])}")

Проверка типа изображения в датасете:
Тип example['image']: <class 'str'>
Декодирование изображений из строк в PIL Image...
После декодирования тип: <class 'PIL.JpegImagePlugin.JpegImageFile'>
Форматирование датасета (без вложенных изображений)...


Map (num_proc=2):   0%|          | 0/1000 [00:00<?, ? examples/s]

Форматирование завершено успешно.
Колонки датасета: ['image', 'messages']
Тип image: <class 'PIL.JpegImagePlugin.JpegImageFile'>


5. Загрузка модели с QLoRA

In [7]:
# Ячейка 7: Загрузка модели с QLoRA
print("Загрузка модели Qwen2-VL-2B с 4-битным квантованием...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa" if torch.cuda.is_available() else "eager"
)

processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-2B-Instruct")
processor.model_max_length = 1024

model = prepare_model_for_kbit_training(model)

print(f"Модель загружена. Параметры: {model.get_input_embeddings().num_embeddings}")

Загрузка модели Qwen2-VL-2B с 4-битным квантованием...


config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/56.4k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Модель загружена. Параметры: 151936


6. Настройка LoRA

In [8]:
# Ячейка 8: Настройка LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 9,232,384 || all params: 2,218,217,984 || trainable%: 0.4162


7. Настройка параметров обучения

In [9]:
# Ячейка 9: Настройка параметров обучения
training_args = TrainingArguments(
    output_dir="./qwen2vl-2b-ru-lora",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    fp16=False,
    bf16=True,
    logging_steps=10,
    save_strategy="epoch",
    optim="paged_adamw_32bit",
    gradient_checkpointing=True,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    report_to="none",
    save_total_limit=1,
    dataloader_num_workers=2,
    remove_unused_columns=False,
)

print(f"Параметры обучения настроены")
print(f"Эпох: {training_args.num_train_epochs}")
print(f"Batch size: {training_args.per_device_train_batch_size}")
print(f"Gradient accumulation: {training_args.gradient_accumulation_steps}")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Параметры обучения настроены
Эпох: 1
Batch size: 2
Gradient accumulation: 4


8. Создание Trainer и обучение

In [12]:
# Ячейка 10: Создание Trainer и обучение
def data_collator(features):
    batch_images = []
    batch_messages = []

    for f in features:
        img = f["image"]
        batch_images.append(img)

        msgs = f["messages"]
        for msg in msgs:
            if msg["role"] == "user":
                original_text = msg["content"]
                msg["content"] = [
                    {"type": "image", "image": img},
                    {"type": "text", "text": original_text}
                ]
                break

        text = processor.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=False
        )
        batch_messages.append(text)

    batch = processor(
        text=batch_messages,
        images=batch_images,
        return_tensors="pt",
        padding=True
    )

    labels = batch["input_ids"].clone()

    for i, feature in enumerate(features):
        prompt_msgs = [feature["messages"][0]]
        prompt_msgs[0]["content"] = [
            {"type": "image", "image": feature["image"]},
            {"type": "text", "text": feature["messages"][0]["content"][1]["text"]}
        ]
        prompt_text = processor.apply_chat_template(
            prompt_msgs,
            tokenize=False,
            add_generation_prompt=True
        )
        prompt_len = len(processor.tokenizer(prompt_text, add_special_tokens=False).input_ids)
        labels[i, :prompt_len] = -100

    labels[batch["attention_mask"] == 0] = -100
    batch["labels"] = labels

    return batch

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_formatted,
    data_collator=data_collator,
)

print("Начало обучения...")
trainer.train()

Начало обучения...


[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
10,9.866089
20,5.361624
30,4.959892
40,4.877368
50,4.811542
60,4.766238
70,4.867150
80,4.772573
90,4.841543
100,4.867770


TrainOutput(global_step=125, training_loss=5.297407409667969, metrics={'train_runtime': 6648.027, 'train_samples_per_second': 0.15, 'train_steps_per_second': 0.019, 'total_flos': 8527557968928768.0, 'train_loss': 5.297407409667969, 'epoch': 1.0})

9. Сохранение модели

In [13]:
# Ячейка 11: Сохранение модели
output_dir = "./qwen2vl-2b-ru-lora-final"

model.save_pretrained(output_dir)
processor.save_pretrained(output_dir)

print(f"LoRA адаптер сохранен в {output_dir}")

total_size = sum(os.path.getsize(os.path.join(output_dir, f)) for f in os.listdir(output_dir) if os.path.isfile(os.path.join(output_dir, f)))
print(f"Размер сохраненных файлов: {total_size / 1e6:.2f} MB")

LoRA адаптер сохранен в ./qwen2vl-2b-ru-lora-final
Размер сохраненных файлов: 48.42 MB


10. Оценка на MMBench-ru

In [14]:
# Ячейка 12: Функция оценки (исправленная)
def evaluate_mmbench(model, processor, dataset, max_samples=100, model_name="Model"):
    if max_samples:
        dataset = dataset.select(range(max_samples))

    correct = 0
    total = 0
    skipped = 0
    model.eval()

    print(f"Начало оценки {model_name} на {len(dataset)} примерах...")

    for i, example in enumerate(dataset):
        if i % 20 == 0 and i > 0:
            print(f"Обработано: {i}/{len(dataset)}, Верно: {correct}, Пропущено: {skipped}")

        question = example["question"]
        hint = example.get("hint", None)

        options = {
            "A": example.get("A", ""),
            "B": example.get("B", ""),
            "C": example.get("C", ""),
            "D": example.get("D", "")
        }
        answer = example["answer"]

        options_text = "\n".join([f"{key}. {val}" for key, val in options.items() if val])

        prompt = f"{question}\n"
        if hint and str(hint).strip() != "nan" and str(hint).strip() != "":
            prompt += f"Подсказка: {hint}\n"
        prompt += f"\nВарианты ответов:\n{options_text}\n\nОтветь только одной буквой (A, B, C или D)."

        try:
            image = example["image"]
            if isinstance(image, str):
                if os.path.exists(image):
                    image = Image.open(image).convert("RGB")
                else:
                    image = Image.open(BytesIO(requests.get(image).content)).convert("RGB")
        except Exception as e:
            print(f"Ошибка загрузки изображения: {e}")
            skipped += 1
            continue

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": prompt}
                ]
            }
        ]

        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], images=[image], return_tensors="pt", padding=True)

        # ИСПРАВЛЕНО: переносим на правильное устройство
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=10,
                do_sample=False,
                temperature=0.1,
                pad_token_id=processor.tokenizer.eos_token_id
            )

        generated_text = processor.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

        # ИСПРАВЛЕНО: более надежный парсинг буквы
        generated_answer = None
        generated_text_upper = generated_text.upper()

        # Ищем букву как отдельное слово
        match = re.search(r'\b([A-D])\b', generated_text_upper)
        if match:
            generated_answer = match.group(1)
        else:
            # Если не нашли как слово, ищем первую встреченную букву
            for letter in ["A", "B", "C", "D"]:
                if letter in generated_text_upper:
                    generated_answer = letter
                    break

        true_answer = str(answer).strip().upper().replace(".", "")
        if generated_answer == true_answer:
            correct += 1
        total += 1

        if max_samples and total >= max_samples:
            break

    accuracy = correct / total if total > 0 else 0
    return accuracy, correct, total

11. Запуск оценки базовой модели

In [17]:
# Ячейка 13: Оценка базовой модели (baseline)
from peft import PeftModel
print("="*60)
print("ОЦЕНКА БАЗОВОЙ МОДЕЛИ (без дообучения)")
print("="*60)

# Загружаем базовую модель без LoRA
model_base = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

base_accuracy, base_correct, base_total = evaluate_mmbench(
    model_base,
    processor,
    eval_dataset,
    max_samples=100,
    model_name="Базовая модель (baseline)"
)

print(f"\n{'='*60}")
print(f"РЕЗУЛЬТАТЫ БАЗОВОЙ МОДЕЛИ")
print(f"{'='*60}")
print(f"Обработано примеров: {base_total}")
print(f"Правильных ответов: {base_correct}")
print(f"Итоговая Accuracy: {base_accuracy * 100:.2f}%")
print(f"{'='*60}")

# Освобождаем память
del model_base
torch.cuda.empty_cache()

ОЦЕНКА БАЗОВОЙ МОДЕЛИ (без дообучения)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Начало оценки Базовая модель (baseline) на 100 примерах...
Обработано: 20/100, Верно: 13, Пропущено: 0
Обработано: 40/100, Верно: 26, Пропущено: 0
Обработано: 60/100, Верно: 38, Пропущено: 0
Обработано: 80/100, Верно: 48, Пропущено: 0

РЕЗУЛЬТАТЫ БАЗОВОЙ МОДЕЛИ
Обработано примеров: 100
Правильных ответов: 58
Итоговая Accuracy: 58.00%


11. Запуск оценки дообученной модели

In [18]:
# Ячейка 14: Оценка дообученной модели
print("="*60)
print("ОЦЕНКА ДООбУЧЕННОЙ МОДЕЛИ (с LoRA)")
print("="*60)

# Загружаем модель с LoRA адаптером
model_finetuned = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

model_finetuned = PeftModel.from_pretrained(model_finetuned, output_dir)
model_finetuned = model_finetuned.merge_and_unload()

finetuned_accuracy, finetuned_correct, finetuned_total = evaluate_mmbench(
    model_finetuned,
    processor,
    eval_dataset,
    max_samples=100,
    model_name="Дообученная модель (LoRA)"
)

print(f"\n{'='*60}")
print(f"РЕЗУЛЬТАТЫ ДООбУЧЕННОЙ МОДЕЛИ")
print(f"{'='*60}")
print(f"Обработано примеров: {finetuned_total}")
print(f"Правильных ответов: {finetuned_correct}")
print(f"Итоговая Accuracy: {finetuned_accuracy * 100:.2f}%")
print(f"{'='*60}")

ОЦЕНКА ДООбУЧЕННОЙ МОДЕЛИ (с LoRA)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:373: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Начало оценки Дообученная модель (LoRA) на 100 примерах...
Обработано: 20/100, Верно: 13, Пропущено: 0
Обработано: 40/100, Верно: 25, Пропущено: 0
Обработано: 60/100, Верно: 37, Пропущено: 0
Обработано: 80/100, Верно: 46, Пропущено: 0

РЕЗУЛЬТАТЫ ДООбУЧЕННОЙ МОДЕЛИ
Обработано примеров: 100
Правильных ответов: 56
Итоговая Accuracy: 56.00%


In [19]:
# Ячейка 15: Сравнение результатов
print("\n" + "="*60)
print("СРАВНЕНИЕ РЕЗУЛЬТАТОВ")
print("="*60)
print(f"Базовая модель:      {base_accuracy * 100:.2f}% ({base_correct}/{base_total})")
print(f"Дообученная модель:  {finetuned_accuracy * 100:.2f}% ({finetuned_correct}/{finetuned_total})")
print(f"Разница:             {(finetuned_accuracy - base_accuracy) * 100:+.2f}%")
print("="*60)

# Сохраняем результаты
results = {
    "baseline_accuracy_percent": round(base_accuracy * 100, 2),
    "baseline_correct": base_correct,
    "baseline_total": base_total,
    "finetuned_accuracy_percent": round(finetuned_accuracy * 100, 2),
    "finetuned_correct": finetuned_correct,
    "finetuned_total": finetuned_total,
    "improvement_percent": round((finetuned_accuracy - base_accuracy) * 100, 2),
    "base_model": "Qwen/Qwen2-VL-2B-Instruct",
    "method": "QLoRA (4-bit, r=8, alpha=16)",
    "train_samples_used": NUM_TRAIN_SAMPLES,
    "training_device": "Tesla T4" if torch.cuda.is_available() else "CPU",
    "training_epochs": 1,
    "learning_rate": 2e-4,
    "batch_size": training_args.per_device_train_batch_size,
    "gradient_accumulation_steps": training_args.gradient_accumulation_steps
}

with open("evaluation_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"\nРезультаты сохранены в evaluation_results.json")


СРАВНЕНИЕ РЕЗУЛЬТАТОВ
Базовая модель:      58.00% (58/100)
Дообученная модель:  56.00% (56/100)
Разница:             -2.00%

Результаты сохранены в evaluation_results.json


In [20]:
# Ячейка 16: Примеры генерации
print("\n" + "="*60)
print("ПРИМЕРЫ ГЕНЕРАЦИИ")
print("="*60)

model_finetuned.eval()
test_samples = eval_dataset.select(range(5))

for i, example in enumerate(test_samples):
    print(f"\n--- Пример {i+1} ---")
    print(f"Вопрос: {example['question']}")

    image = example["image"]
    if isinstance(image, str) and os.path.exists(image):
        image = Image.open(image).convert("RGB")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": example['question']}
            ]
        }
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors="pt", padding=True)
    inputs = {k: v.to(model_finetuned.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model_finetuned.generate(
            **inputs,
            max_new_tokens=50,
            do_sample=True,
            temperature=0.7
        )

    response = processor.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"Ответ модели: {response}")
    print(f"Правильный ответ: {example['answer']}")


ПРИМЕРЫ ГЕНЕРАЦИИ

--- Пример 1 ---
Вопрос: Какая часть яблони может вырасти в новое дерево?
Ответ модели: Яблоня может вырасти в новое дерево, если она будет опылена пчелами.
Правильный ответ: A

--- Пример 2 ---
Вопрос: Из какого материала сделана эта лопатка?
Ответ модели: Эта лопатка сделана из силикона.
Правильный ответ: A

--- Пример 3 ---
Вопрос: Это место многолюдное?
Ответ модели: Извините, но я не могу помочь с этим.
Правильный ответ: A

--- Пример 4 ---
Вопрос: Какое изображение более красочное?
Ответ модели: Красочнее выглядит изображение на левой стороне, где видны детали интерьера дома и окружение.
Правильный ответ: A

--- Пример 5 ---
Вопрос: Какая картинка ярче?
Ответ модели: Вот две изображения:

1. Слева: красный светофор.
2. Справа: изображение черепа в форме мордашки.

Какая из этих картинок ярче?
Правильный ответ: A


In [21]:
# Ячейка 17: Финальная информация
print("\n" + "="*60)
print("ПРОЕКТ ЗАВЕРШЕН")
print("="*60)
print(f"Обучено на: {NUM_TRAIN_SAMPLES} примерах")
print(f"Оценено на: {finetuned_total} примерах")
print(f"Baseline accuracy: {base_accuracy * 100:.2f}%")
print(f"Finetuned accuracy: {finetuned_accuracy * 100:.2f}%")
print(f"Улучшение: {(finetuned_accuracy - base_accuracy) * 100:+.2f}%")
print(f"Модель сохранена в: {output_dir}")
print(f"Результаты сохранены в: evaluation_results.json")
print("="*60)


ПРОЕКТ ЗАВЕРШЕН
Обучено на: 1000 примерах
Оценено на: 100 примерах
Baseline accuracy: 58.00%
Finetuned accuracy: 56.00%
Улучшение: -2.00%
Модель сохранена в: ./qwen2vl-2b-ru-lora-final
Результаты сохранены в: evaluation_results.json
